In [1]:

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from finance_byu.summarize import summary
from finance_byu.regtables import Regtable
import statsmodels.formula.api as smf
from scipy import stats

In [2]:
# HELPER FUNCTIONS
def CheckContinuity(df):
    full_range = pd.date_range(df['Date'].min(), df['Date'].max(),freq='MS')
    # Checking for gaps in dates that could throw off momentum
    return df['Date'].nunique() == len(full_range)


def LoadCleanFile(filename):
    df = pd.read_csv(filename,na_values=[-99.99,-999])

    df['Date'] = df['Date'].astype(str)

    # eliminate yearly data
    df['Date'] = df['Date'][df['Date'].apply(lambda x: len(x)) == 6]
    # Get rid of nans
    df = df.query('Date == Date')

    # convert to datetime
    df['Date'] = pd.to_datetime(df['Date'], format='%Y%m').values.astype("datetime64[M]")
    if len(df) != df.Date.nunique():
        df = df.drop_duplicates(subset='Date')

    
    if not CheckContinuity(df):
        print("WARNING: date range is not continuous.")

    return df.dropna()

# load in factor datasets
F3 = LoadCleanFile('FF3_factor_data.csv')
F5 = LoadCleanFile('FF5_factor_data.csv')

def CreateRollingIndex(df,window_size=6):
    start = df.Date.min()
    n_windows = df.Date.nunique()

    idx = pd.DatetimeIndex(
    pd.concat([
        pd.Series(pd.date_range(start + pd.DateOffset(months=i),
                                periods=window_size, freq="MS"))
        for i in range(n_windows)
    ]))

    # make sure the last date where we can hold for 6 months is the last porfolio
    return idx[:-(window_size*(window_size-1))]

def RunReturnAnalysis(df, start='1963-07', end='1995-07'):
    cols = df.columns[1:]
    df.iloc[:,1:] = df.iloc[:,1:].apply(pd.to_numeric, errors='coerce')
    mom_dict = {'Date':df['Date']}
    # creating momentum for each industry
    for name in cols:
        # I originally did this wrong, should be the cummulative product as outlined in the 
        # Jagadeesh and Titman paper. They normally shift 1 but we shifted 2 in class so I'm doing that.
        mom_dict[f"lagmom_{name}"] = df[name].transform(lambda x: 1 + x/100).shift(2).rolling(6,6).apply(lambda x: x.prod() - 1)
    # dropping where not enough values to get 6 full months of data
    mom_df = pd.DataFrame(mom_dict)
    mom_df = mom_df.dropna().reset_index(drop=True)

    # get the sample period
    df = df[(df['Date'] >= start) & (df['Date'] <= end)]
    mom_df = mom_df[(mom_df['Date'] >= start) & (mom_df['Date'] <= end)]


    if len(df) > len(mom_df):
        print('entered')
        df = df.loc[df.Date.isin(mom_df.Date)]
    elif len(df) < len(mom_df):
        mom_df = mom_df.loc[mom_df.Date.isin(df.Date)]

    df = df.sort_values('Date')
    mom_df = mom_df.sort_values('Date')

    if not CheckContinuity(df):
        print("WARNING: date range is not continuous in original dataframe.")
    if not CheckContinuity(mom_df):
        print("WARNING: date range is not continuous in momentum dataframe.")


    # get array of lagged industry momentum values to get portfolio choices
    # and then the array of returns to construct the portfolios
    array = mom_df.iloc[:,1:].values.astype(float)
    ret_array = df.iloc[:,1:].values.astype(float)


    hold_val = 6 # how long we'll hold each portfolio
    sorter = np.argsort(array,axis=1)
    # get choices per time period
    top3 = sorter[:-hold_val+1,-3:]
    num_buys = len(top3) # I need the total number of portfolios I will buy and hold for six months here.
    top3 = np.array([np.repeat(a.reshape(1,-1), hold_val, axis=0) for a in top3]).reshape(-1,3)
    bot3 = sorter[:-hold_val+1,:3]
    bot3 = np.array([np.repeat(a.reshape(1,-1), hold_val, axis=0) for a in bot3]).reshape(-1,3)


    # get all values that will represent profits for 6-month held portfolios starting the day lagged momentum was decided
    broadcaster = np.array([i+j for i in range(num_buys) for j in range(hold_val)]).reshape(-1,1)
    returns = ret_array[broadcaster,top3].mean(axis=1) - ret_array[broadcaster,bot3].mean(axis=1)


    # create rolling index to encompass multiple portfolios on multiple days.
    window_dates = CreateRollingIndex(df)
    port_rets = pd.DataFrame({'Date':window_dates,'part_ret':returns})

    results = (port_rets.groupby('Date')['part_ret'].mean().to_frame(name='ret'))

    print(summary(results))
    return results
    # return results.to_frame(name='ret')


def RunRegressionAnalysis(results):
    # FF3 Regression
    # merge portfolio results to regress on
    reg_df = pd.merge(results, F3, on='Date')
    # get return over and above risk free rate
    reg_df['ret_oa'] = reg_df['ret'] - reg_df['RF']
    # fix problem with variable name (smf.old thought it was a minus sign)
    reg_df['Mkt'] = reg_df['Mkt-RF']

    # FF5 Regression
    reg1 = smf.ols('ret_oa ~ 1 + Mkt + SMB + HML',data=reg_df).fit()

    reg_df = pd.merge(results, F5, on='Date')
    reg_df['ret_oa'] = reg_df['ret'] - reg_df['RF']
    reg_df['Mkt'] = reg_df['Mkt-RF']

    reg2 = smf.ols('ret_oa ~ 1 + Mkt + SMB + HML + RMW + CMA',data=reg_df).fit()

    tbl = Regtable([reg1,reg2],stat='tstat',sig='coeff')
    return tbl.render()



# NOTES:
"""
According to Part B of the paper the momentum is calculated in the last 6 months and portolfios are created by investing in the top 3 momentum industries
and shorting the bottom 3 and holding for a period of 6 months. (this is for industries, it's top and bottom 30% for individual stocks)
"""

"\nAccording to Part B of the paper the momentum is calculated in the last 6 months and portolfios are created by investing in the top 3 momentum industries\nand shorting the bottom 3 and holding for a period of 6 months. (this is for industries, it's top and bottom 30% for individual stocks)\n"

In [3]:
# start='1963-07'
# end='1995-07'
start='1995-08'
end='2026-01'

# 17 Firms

In [4]:
df = LoadCleanFile('17_Industry_Portfolios.csv')
results17 = RunReturnAnalysis(df,start=start,end=end)

RunRegressionAnalysis(results17)

              ret
count  366.000000
mean     0.428063
std      4.520964
tstat    1.811411
pval     0.070899
min    -15.678889
25%     -2.070694
50%      0.445556
75%      3.085972
max     15.451667


,ret_oa,ret_oa
Intercept,0.339,0.237
,(1.45),(0.99)
Mkt,-0.084,-0.033
,(-1.60),(-0.58)
SMB,0.097,0.111
,(1.30),(1.32)
HML,-0.299***,-0.479***
,(-4.30),(-4.97)
RMW,,0.074
,,(0.69)


# 30 Firms

In [5]:
df = LoadCleanFile('30_Industry_Portfolios.csv')
results30 = RunReturnAnalysis(df,start=start,end=end)

RunRegressionAnalysis(results30)

              ret
count  366.000000
mean     0.684976
std      5.698674
tstat    2.299546
pval     0.022038
min    -21.435556
25%     -2.525000
50%      0.429722
75%      3.857500
max     19.732778


,ret_oa,ret_oa
Intercept,0.659**,0.506
,(2.22),(1.67)
Mkt,-0.164**,-0.090
,(-2.45),(-1.24)
SMB,0.055,0.086
,(0.58),(0.80)
HML,-0.310***,-0.549***
,(-3.50),(-4.49)
RMW,,0.135
,,(1.00)


# 48 Firms

In [6]:
df = LoadCleanFile('48_Industry_Portfolios_copy2.csv')
results48 = RunReturnAnalysis(df,start=start,end=end)

RunRegressionAnalysis(results48)

              ret
count  366.000000
mean     0.694528
std      6.514171
tstat    2.039722
pval     0.042098
min    -28.868889
25%     -2.784028
50%      0.865556
75%      4.053194
max     20.390556


,ret_oa,ret_oa
Intercept,0.702**,0.505
,(2.06),(1.45)
Mkt,-0.208***,-0.121
,(-2.71),(-1.45)
SMB,0.039,0.115
,(0.36),(0.94)
HML,-0.305***,-0.571***
,(-3.01),(-4.08)
RMW,,0.242
,,(1.55)


# Mean and Tstat Comparisons

In [7]:
print(f"{"":15} {"Mean":>8} {"Tstat":>8}")
print(f"{"17 Industries":15} {results17['ret'].mean():>8.3f} {results17['ret'].mean() / (results17['ret'].std() / np.sqrt(len(results17))):>8.3}")
print(f"{"30 Industries":15} {results30['ret'].mean():>8.3f} {results30['ret'].mean() / (results30['ret'].std() / np.sqrt(len(results30))):>8.3}")
print(f"{"48 Industries":15} {results48['ret'].mean():>8.3f} {results48['ret'].mean() / (results48['ret'].std() / np.sqrt(len(results48))):>8.3}")

                    Mean    Tstat
17 Industries      0.428     1.81
30 Industries      0.685      2.3
48 Industries      0.695     2.04
